In [1]:
# pytorch
#!pip install transformers==4.20.0
#!pip install -e git+https://github.com/kpu/kenlm.git#egg=kenlm
#!pip install pyctcdecode==0.4.0
#!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu116
import os
from importlib.machinery import SourceFileLoader

import torch
import torchaudio
from pydub import AudioSegment
from transformers import Wav2Vec2ProcessorWithLM
from transformers.file_utils import cached_path, hf_bucket_url

os.chdir("/home/gpu/fuzzy")

print(os.getcwd())

/home/gpu/.cache/pypoetry/virtualenvs/traffic-congestion-No1flJwg-py3.9/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/gpu/fuzzy


In [2]:
# Kenlm gcc fix: https://github.com/stan-dev/pystan/issues/294

In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)

cuda:0


In [4]:
# Trim audio
voh_record = AudioSegment.from_wav("95.6_2022_11_30_104500_104515.wav")
clipped_vod_record = voh_record[: 5 * 1000]
clipped_vod_record.export("voh_record.wav", format="wav")

<_io.BufferedRandom name='voh_record.wav'>

In [5]:
# Load model & processor
model_name = "nguyenvulebinh/wav2vec2-large-vi-vlsp2020"
model = (
    SourceFileLoader(
        "model", cached_path(hf_bucket_url(model_name, filename="model_handling.py"))
    )
    .load_module()
    .Wav2Vec2ForCTC.from_pretrained(model_name)
)
model = model.to(device)
model.zero_grad()
processor = Wav2Vec2ProcessorWithLM.from_pretrained(model_name)

# Load an example audio (16k)
# audio, sample_rate = torchaudio.load(
#     cached_path(hf_bucket_url(model_name, filename="t2_0000006682.wav"))
# )
audio, sample_rate = torchaudio.load("voh_record.wav")
print(sample_rate)

/home/gpu/.cache/pypoetry/virtualenvs/traffic-congestion-No1flJwg-py3.9/lib/python3.9/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in 'snapshot_download': allow_regex. Will not be supported from version '0.12'.

Please use `allow_patterns` and `ignore_patterns` instead.
  warnings.warn(message, FutureWarning)
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 10255.02it/s]
Only 0 unigrams passed as vocabulary. Is this small or artificial data?


44100


In [6]:
input_data = processor.feature_extractor(
    audio[0], sampling_rate=16000, return_tensors="pt"
).to(device)

# Infer
output = model(**input_data)

# Output transcript without LM
print(
    processor.tokenizer.decode(output.logits.argmax(dim=-1)[0].detach().cpu().numpy())
)

# Output transcript with LM
print(processor.decode(output.logits.cpu().detach().numpy()[0], beam_width=100).text)

thơ chưa ờn đt đàn ờ ở xơ ưa hơ sức hạn ở am và
thơ chưa tàn xơ sức hạn ở an và
